<a href="https://colab.research.google.com/github/atanasovmi/Quantum-/blob/main/notebookB_picture_classifier_LA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Picture classifier

In [ ]:
!pip install qiskit qiskit-aer qiskit-machine-learning medmnist pylatexenc qiskit-optimization


In [ ]:
!pip install qiskit-algorithms

In [ ]:
from medmnist import BreastMNIST
from skimage.transform import resize
import numpy as np

def load_and_preprocess_data(split='train', size=(4, 4)):
    # Load BreastMNIST dataset
    dataset = BreastMNIST(split=split, download=True)

    # Preprocess images
    preprocessed_images = []
    for i in range(len(dataset)):
        # Resize the image to match the quantum circuit size
        image = resize(np.array(dataset[i][0]), size, anti_aliasing=True)

        # Flatten the image to a 1D array and normalize pixel values
        flattened_image = image.flatten() / 255.0

        # Append the processed image data to the preprocessed_images list
        preprocessed_images.append(flattened_image)

    # Extract labels and flatten if necessary
    labels = np.array([dataset[i][1] for i in range(len(dataset))]).astype(int)
    if labels.ndim > 1:
        labels = labels.flatten()

    return np.array(preprocessed_images), labels


In [ ]:
# Load and preprocess the data
train_data, train_labels = load_and_preprocess_data(split='train', size=(3, 3))
test_data, test_labels = load_and_preprocess_data(split='test', size=(3, 3))

# Verify the shapes of the data and labels
print("train_data.shape:", train_data.shape)
print("train_labels.shape:", train_labels.shape)
print("Unique labels:", np.unique(train_labels))


In [ ]:
# Reduce dataset size for faster training
train_data = train_data[:50]
train_labels = train_labels[:50]
test_data = test_data[:20]
test_labels = test_labels[:20]

print(f"\n=== Reduced Dataset ===")
print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
# ================================================

Classical approach

In [ ]:
# Import scikit-learn components
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Create a pipeline with a scaler and an SVC
svc = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

# Fit the SVM model
svc.fit(train_data, train_labels)

# Evaluate the SVM model
svc_score = svc.score(test_data, test_labels)
print(f'SVC accuracy: {svc_score:.2f}')


Quantum approach

In [ ]:
from qiskit.circuit.library import ZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler as Sampler  # Updated for newer Qiskit
from matplotlib import pyplot as plt
from IPython.display import clear_output

In [ ]:
# Set up the feature map using the correct number of features
num_features = train_data.shape[1]
feature_map = ZFeatureMap(feature_dimension=num_features, reps=1)

# Set up the ansatz
ansatz = RealAmplitudes(num_qubits=num_features, reps=3)

# Initialize the optimizer
optimizer = COBYLA(maxiter=10)

# Initialize the sampler
sampler = Sampler()

# Callback function for visualizing optimization progress
objective_func_vals = []
plt.rcParams["figure.figsize"] = (12, 6)

def callback_graph(weights, obj_func_eval):
    clear_output(wait=True)
    objective_func_vals.append(obj_func_eval)
    plt.title("Objective function value against iteration")
    plt.xlabel("Iteration")
    plt.ylabel("Objective function value")
    plt.plot(range(len(objective_func_vals)), objective_func_vals)
    plt.show()


In [ ]:
import time
from qiskit_machine_learning.algorithms.classifiers import VQC

# Initialize the VQC classifier
vqc = VQC(
    sampler=sampler,
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=optimizer,
    callback=callback_graph,
)

# Clear objective value history
objective_func_vals = []

# Train the VQC model
start = time.time()
vqc.fit(train_data, train_labels)
elapsed = time.time() - start

print(f"Training time: {round(elapsed)} seconds")


In [ ]:
# Evaluate the VQC model
vqc_score = vqc.score(test_data, test_labels)
print(f'VQC accuracy: {vqc_score:.2f}')


In [ ]:
vqc.circuit.decompose().draw("mpl")
